<a href="https://colab.research.google.com/github/pranavik24/PromptInject/blob/main/notebooks/comparisons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install sentence-transformers
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=cad96b0b76bd3e52339dc9a6b0d8993fb2a82823e93ec012fc98fa07a404766b
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [14]:
import pandas as pd
import re
import math
from sentence_transformers import SentenceTransformer
import numpy as np
from rouge_score import rouge_scorer

In [2]:
# Cell 1 — mount google drive
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [3]:
model = SentenceTransformer("all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:

def encode_outputs(outputs1: list[str], outputs2: list[str], batch_size: int = 32):
    vecs1 = model.encode(outputs1, batch_size=batch_size, show_progress_bar=True)
    vecs2 = model.encode(outputs2, batch_size=batch_size, show_progress_bar=True)
    return vecs1, vecs2

In [8]:

def compute_cosine_scores(vecs1: np.ndarray, vecs2: np.ndarray) -> np.ndarray:
    norms1 = np.linalg.norm(vecs1, axis=1, keepdims=True)
    norms2 = np.linalg.norm(vecs2, axis=1, keepdims=True)
    return np.round(np.sum((vecs1 / norms1) * (vecs2 / norms2), axis=1), 4)

In [15]:
def compute_rouge_scores(outputs1: list[str], outputs2: list[str]) -> tuple[list, list]:
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2"], use_stemmer=True)
    rouge1_scores, rouge2_scores = [], []
    for o1, o2 in zip(outputs1, outputs2):
        scores = scorer.score(o1, o2)
        rouge1_scores.append(round(scores["rouge1"].fmeasure, 4))
        rouge2_scores.append(round(scores["rouge2"].fmeasure, 4))
    return rouge1_scores, rouge2_scores

In [16]:
def compare_model_outputs(path_without_cot: str, path_with_cot: str, output_path: str):
    df1 = pd.read_csv(path_without_cot)
    df2 = pd.read_csv(path_with_cot)

    assert len(df1) == len(df2), "CSVs have different number of rows"

    outputs1 = df1["output"].astype(str).tolist()
    outputs2 = df2["output"].astype(str).tolist()

    print("Encoding outputs...")
    vecs1, vecs2 = encode_outputs(outputs1, outputs2)

    print("Computing cosine similarity scores...")
    cosine_scores = compute_cosine_scores(vecs1, vecs2)

    print("Computing ROUGE scores...")
    rouge1_scores, rouge2_scores = compute_rouge_scores(outputs1, outputs2)

    pd.DataFrame({
        "number": df1["number"],
        "prompt_instruction": df1["prompt_instruction"],
        "attack_instruction": df1["attack_instruction"],
        "output_WithoutCOT": df1["output"],
        "output_WithCOT": df2["output"],
        "cosineSimilarityScore": cosine_scores,
        "rouge1": rouge1_scores,
        "rouge2": rouge2_scores,
    }).to_csv(output_path, index=False)
    print(f"Done. Results saved to {output_path}")

In [18]:
compare_model_outputs(
    path_without_cot="/content/drive/MyDrive/11711 group project/injection_withoutCOT.csv",
    path_with_cot="/content/drive/MyDrive/11711 group project/injection_withCOT.csv",
    output_path="/content/drive/MyDrive/11711 group project/comparison_results.csv",
)

Encoding outputs...


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Computing cosine similarity scores...
Computing ROUGE scores...
Done. Results saved to /content/drive/MyDrive/11711 group project/comparison_results.csv
